# 06a — Autoencoder Basics

## The One-Sentence Version
Squeeze data through a bottleneck and try to reconstruct it — whatever survives the bottleneck is the compressed representation.

## What You'll Build Intuition For
- The encoder-bottleneck-decoder architecture
- How a neural network learns to compress data
- How the latent space compares to PCA
- How bottleneck size controls the quality-compression trade-off

## Prerequisites
- [03b — PCA from Scratch](../03_linear_methods/03b_pca_from_scratch.ipynb) — the linear version of compression
- [05c — Embedded Methods](../05_feature_selection/05c_embedded_methods.ipynb) — models that learn what matters

## The Story

Imagine you need to describe a photograph over a walkie-talkie. You can't send the pixels — you have to compress the image into a few words. If you're good at it, the listener can reconstruct a reasonable mental picture from your description.

An autoencoder does exactly this with a neural network. It has three parts:
1. **Encoder**: compresses the input into a small representation (the "description")
2. **Bottleneck**: the compressed representation itself (the "walkie-talkie channel")
3. **Decoder**: reconstructs the input from the compressed version (the "listener")

The network is trained end-to-end: minimize the difference between input and reconstruction. Whatever information survives the bottleneck is what the network decided was most important.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_digit_data

apply_style()
rng = np.random.default_rng(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## The Architecture

```
Input (64)  →  [Encoder: 64→128→latent]  →  Bottleneck (2)  →  [Decoder: latent→128→64]  →  Output (64)
```

The hidden layer *is* the compressed version. If the bottleneck has 2 neurons and the input has 64, the network must learn to keep only what's essential.

In [ ]:
class SimpleAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

    def encode(self, x):
        return self.encoder(x)

# Architecture diagram
fig, ax = plt.subplots(figsize=(12, 4))
layers = [64, 128, 2, 128, 64]
labels = ["Input\n(64D)", "Hidden\n(128)", "Bottleneck\n(2D)", "Hidden\n(128)", "Output\n(64D)"]
x_pos = np.linspace(0.1, 0.9, len(layers))

for i, (x, size, label) in enumerate(zip(x_pos, layers, labels)):
    height = size / 140  # scale
    colour = COLOURS["signal"] if i == 2 else COLOURS["neutral"]
    if i < 2:
        colour = COLOURS["accent"]
    elif i > 2:
        colour = COLOURS["success"]
    rect = plt.Rectangle((x - 0.03, 0.5 - height/2), 0.06, height,
                          facecolor=colour, alpha=0.7, edgecolor="white", linewidth=2)
    ax.add_patch(rect)
    ax.text(x, -0.05, label, ha="center", fontsize=10, fontweight="bold")
    if i < len(layers) - 1:
        ax.annotate("", xy=(x_pos[i+1] - 0.035, 0.5), xytext=(x + 0.035, 0.5),
                    arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"], lw=1.5))

ax.text(0.3, 0.95, "Encoder", ha="center", fontsize=12, fontweight="bold",
        color=COLOURS["accent"], transform=ax.transAxes)
ax.text(0.7, 0.95, "Decoder", ha="center", fontsize=12, fontweight="bold",
        color=COLOURS["success"], transform=ax.transAxes)

ax.set_xlim(0, 1)
ax.set_ylim(-0.15, 1.1)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Autoencoder Architecture: squeeze through a bottleneck")
plt.tight_layout()
plt.savefig("visuals/06a_architecture.png", dpi=150, bbox_inches="tight")
plt.show()

## Train on Digits

We'll compress the 64-dimensional digit images down to just 2 dimensions. The network must learn which features of each digit are essential for reconstruction.

In [ ]:
# Prepare data
data, target, images = make_digit_data()
X_tensor = torch.FloatTensor(data / 16.0).to(device)  # normalise to [0, 1]
dataset = TensorDataset(X_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

def train_autoencoder(latent_dim, epochs=100, lr=1e-3):
    """Train an autoencoder and return model + loss history."""
    model = SimpleAutoencoder(64, latent_dim).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    losses = []

    for epoch in range(epochs):
        epoch_loss = 0
        for (batch,) in loader:
            recon = model(batch)
            loss = criterion(recon, batch)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item() * batch.size(0)
        losses.append(epoch_loss / len(dataset))

    return model, losses

# Train with latent_dim=2
model_2d, losses_2d = train_autoencoder(latent_dim=2, epochs=150)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses_2d, color=COLOURS["signal"], linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training loss: the network learns to compress digits to 2D")
plt.tight_layout()
plt.savefig("visuals/06a_training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Final loss: {losses_2d[-1]:.4f}")

In [ ]:
# Show original vs reconstructed digits
model_2d.eval()
with torch.no_grad():
    X_recon = model_2d(X_tensor).cpu().numpy()

fig, axes = plt.subplots(2, 10, figsize=(14, 3))

# Pick one example of each digit
for digit in range(10):
    idx = np.where(target == digit)[0][0]
    axes[0, digit].imshow(data[idx].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    axes[0, digit].set_xticks([])
    axes[0, digit].set_yticks([])
    if digit == 0:
        axes[0, digit].set_ylabel("Original", fontsize=10)

    axes[1, digit].imshow((X_recon[idx] * 16).reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    axes[1, digit].set_xticks([])
    axes[1, digit].set_yticks([])
    if digit == 0:
        axes[1, digit].set_ylabel("Reconstructed", fontsize=10)

fig.suptitle("Original vs Reconstructed (latent_dim=2): lossy but recognisable",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/06a_reconstruction.png", dpi=150, bbox_inches="tight")
plt.show()

print("With only 2 latent dimensions, the reconstructions are blurry but recognisable.")
print("The network learned to preserve the most distinctive features of each digit.")

## Visualise the Latent Space

The 2D bottleneck gives us a natural visualisation. Let's encode all digits and see how the autoencoder organises them — then compare to PCA.

In [ ]:
# Encode all digits
model_2d.eval()
with torch.no_grad():
    Z_ae = model_2d.encode(X_tensor).cpu().numpy()

# PCA for comparison
Z_pca = PCA(n_components=2).fit_transform(data / 16.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for digit in range(10):
    mask = target == digit
    ax1.scatter(Z_pca[mask, 0], Z_pca[mask, 1], s=10, alpha=0.5,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])
    ax2.scatter(Z_ae[mask, 0], Z_ae[mask, 1], s=10, alpha=0.5,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])

ax1.set_title("PCA (2D)")
ax1.set_xticks([])
ax1.set_yticks([])

ax2.set_title("Autoencoder (2D)")
ax2.set_xticks([])
ax2.set_yticks([])
ax2.legend(title="Digit", markerscale=3, fontsize=8, loc="upper right")

fig.suptitle("Latent space comparison: PCA vs Autoencoder", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/06a_latent_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("PCA: linear projection — some overlap between classes.")
print("Autoencoder: nonlinear compression — can learn curved boundaries.")
print("The autoencoder's latent space often shows tighter, more separated clusters.")

## Vary the Bottleneck

How does bottleneck size affect reconstruction quality? More dimensions = better reconstruction but less compression.

In [ ]:
# Train autoencoders at different bottleneck sizes
latent_dims = [2, 5, 10, 20, 32]
ae_errors = []
pca_errors = []

X_np = data / 16.0

for ld in latent_dims:
    # Autoencoder
    model, _ = train_autoencoder(latent_dim=ld, epochs=100)
    model.eval()
    with torch.no_grad():
        recon = model(X_tensor).cpu().numpy()
    ae_mse = np.mean((X_np - recon) ** 2)
    ae_errors.append(ae_mse)

    # PCA
    pca = PCA(n_components=ld)
    Z = pca.fit_transform(X_np)
    X_pca_recon = pca.inverse_transform(Z)
    pca_mse = np.mean((X_np - X_pca_recon) ** 2)
    pca_errors.append(pca_mse)

    print(f"latent_dim={ld:>2}: AE MSE={ae_mse:.4f}, PCA MSE={pca_mse:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(latent_dims, ae_errors, "o-", color=COLOURS["signal"], linewidth=2,
        markersize=8, label="Autoencoder")
ax.plot(latent_dims, pca_errors, "s--", color=COLOURS["accent"], linewidth=2,
        markersize=8, label="PCA")
ax.set_xlabel("Latent / Component Dimensions")
ax.set_ylabel("Reconstruction MSE")
ax.set_title("Reconstruction error vs bottleneck size")
ax.legend()
plt.tight_layout()
plt.savefig("visuals/06a_bottleneck_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nAt very low dimensions (2-5), the autoencoder can beat PCA")
print("because it captures nonlinear structure.")
print("At higher dimensions, PCA catches up — the data is mostly linear.")

In [ ]:
# Show reconstructions at different bottleneck sizes
fig, axes = plt.subplots(len(latent_dims) + 1, 10, figsize=(14, 2 * (len(latent_dims) + 1)))

# Original row
for digit in range(10):
    idx = np.where(target == digit)[0][0]
    axes[0, digit].imshow(data[idx].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    axes[0, digit].set_xticks([])
    axes[0, digit].set_yticks([])
axes[0, 0].set_ylabel("Original", fontsize=9)

# Reconstruction rows
for row, ld in enumerate(latent_dims, 1):
    model, _ = train_autoencoder(latent_dim=ld, epochs=100)
    model.eval()
    with torch.no_grad():
        recon = model(X_tensor).cpu().numpy()

    for digit in range(10):
        idx = np.where(target == digit)[0][0]
        axes[row, digit].imshow((recon[idx] * 16).reshape(8, 8), cmap="gray_r",
                                 vmin=0, vmax=16)
        axes[row, digit].set_xticks([])
        axes[row, digit].set_yticks([])
    axes[row, 0].set_ylabel(f"dim={ld}", fontsize=9)

fig.suptitle("Reconstruction quality improves with wider bottleneck", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/06a_bottleneck_recon.png", dpi=150, bbox_inches="tight")
plt.show()

> **Key insight:** An autoencoder with a linear activation and MSE loss is mathematically equivalent to PCA. The nonlinear activations (ReLU) are what give autoencoders their extra power — they can learn curved compression surfaces that PCA's flat subspaces can't capture.

<details>
<summary><b>The Maths Behind This</b></summary>

**Loss function:** $\mathcal{L} = \frac{1}{n}\sum_{i=1}^n \|\mathbf{x}_i - \text{dec}(\text{enc}(\mathbf{x}_i))\|^2$

The encoder learns a mapping $f: \mathbb{R}^D \to \mathbb{R}^d$ and the decoder learns $g: \mathbb{R}^d \to \mathbb{R}^D$, where $d \ll D$.

**Linear case:** If both encoder and decoder are single linear layers (no activation), the optimal solution spans the same subspace as PCA's top $d$ components. The weights converge to the principal directions (up to rotation).

**Nonlinear case:** With nonlinear activations, the encoder can learn $f(\mathbf{x}) = \sigma(W_2 \sigma(W_1 \mathbf{x} + b_1) + b_2)$ — a composition of affine maps and pointwise nonlinearities. This can approximate any continuous function (universal approximation), so the autoencoder can capture manifold structure that PCA misses.

**Bottleneck as regularisation:** The bottleneck dimension $d$ controls the capacity of the latent space. Too small → underfitting (can't represent the data). Too large → the network learns the identity function (no compression). The sweet spot compresses maximally while preserving important structure.

</details>

## Where This Matters

**Medical imaging:** Autoencoders compress high-resolution scans (MRI, CT) into compact representations for efficient storage and retrieval. A well-trained autoencoder can reduce a 256×256 image to a 64-dimensional vector while preserving diagnostically relevant features.

**Anomaly detection:** Train an autoencoder on "normal" data. When a new sample arrives, measure its reconstruction error. High error = the sample is unusual (the network hasn't learned to compress it). This is widely used in manufacturing quality control and network intrusion detection.

## Feynman Check

1. **What does the bottleneck layer represent?**

2. **Why might an autoencoder with latent_dim=2 give different results than PCA with n_components=2?**

3. **How do you decide the bottleneck size?**

---

The regular autoencoder compresses well, but its latent space can be disorganised. In **06b — Variational Autoencoders**, we'll learn to make the latent space smooth and navigable.